# Week 06 — Multi-touch attribution

**Goal.** Replace last-click with Markov removal effects and Shapley values, and connect the credit back to bid value.

**Deliverable.** An attribution library with three methods, a credit-distribution comparison, and the bidding implication.

**Rough shape of the week.** 2h reading (Diemert) · 6h building · 1h write-up.

---
### Ground rules (they apply every week)

1. **Beat a dumb baseline or it didn't happen.** Logistic regression or the global mean.
   Log the baseline in the same table as the fancy model.
2. **Split by time, never at random.** `split.time_split` — and call
   `split.check_no_leakage` so the assertion, not your memory, enforces it.
3. **Log every run** with `registry.log_result(...)`, including the ones that lost.
   The losing runs are what make the write-up honest.
4. **Write the finding down** in this week's `README.md` while it is fresh.

### Reading

PDFs are in `papers/` next to this notebook — see `papers/README.md`.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

%load_ext autoreload
%autoreload 2

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from adslab import data, metrics, plots, split, registry, encoders, calibration

plots.use_style()
pd.set_option("display.width", 140, "display.max_columns", 60)
print("harness ready")

## Journeys, not impressions

This week the unit of analysis changes. Group by `conversion_id` (impressions competing
for one conversion) and by `uid` (a user's whole timeline). Use
`split.user_grouped_time_split` — a plain time split cuts a journey in half and leaks the
outcome through the user id.

The dataset ships Criteo's own `attribution` column: their last-click assignment. That is
your baseline and your sanity check, not ground truth. There is no ground truth in
attribution; that is what makes it hard and what makes Week 7 necessary.

In [ ]:
df = data.add_attribution_derived(data.load_attribution())

sp = split.user_grouped_time_split(df, "timestamp", "uid")
split.check_no_leakage(df, sp)   # note: approximate under grouping — read the docstring
print(sp)

journeys = (df[df.conversion == 1]
            .sort_values("timestamp")
            .groupby("conversion_id")
            .agg(path=("campaign", list), n=("campaign", "size")))
print(f"{len(journeys):,} converting journeys, median length {journeys.n.median():.0f}, "
      f"{(journeys.n > 1).mean():.1%} are multi-touch")

## 1. Last click

Trivial to implement and it is 80% of the industry. Reproduce Criteo's `attribution`
column with your own rule and check you agree — if you don't, you have misunderstood the
data, and better to find that out now.

In [ ]:
# TODO: assign credit 1.0 to the last click before the conversion, 0 otherwise
# then: agreement rate with df.attribution

## 2. Markov chain removal effect

Model journeys as paths through a Markov chain over channels/campaigns, with absorbing
`(conversion)` and `(null)` states. The credit of channel $c$ is the **removal effect**:
how much total conversion probability drops when you delete $c$ from the graph.

Implementation notes that save an evening:

- You need *non-converting* journeys too, or every path ends in conversion and every
  removal effect is meaningless.
- Higher-order chains (remembering the last 2 steps) fit better and explode
  combinatorially. First order first.
- Removal effects do not sum to 1. Normalise at the end and say that you did.

In [ ]:
# TODO: build transition matrix, compute baseline conversion prob,
# then recompute with each channel removed

## 3. Shapley value

The axiomatically fair allocation: average marginal contribution over all orderings.
Exact computation is $O(2^n)$ over channels, so:

- with few enough distinct campaigns, compute it exactly on the *coalition* form
  (value of a set = conversion rate of journeys containing exactly that set);
- otherwise sample permutations (Monte Carlo Shapley) and report a confidence interval.
  A Shapley value without an error bar, from a sampler, is a number you cannot defend.

In [ ]:
# TODO: coalition value function + exact or sampled Shapley

## 4. Compare the credit distributions

Three methods, one bar chart of credit per campaign. Then the questions worth answering:

- Which campaigns does last-click *systematically* under-credit? (Upper-funnel ones —
  can you show it?)
- How much does total credit move? Is the reallocation big enough to change a budget
  decision, or is this all statistically indistinguishable?

In [ ]:
# fig, ax = plt.subplots(figsize=(9, 4.5))
# ... grouped bars: last-click / markov / shapley
# print(plots.save(fig, 6, "credit_by_method"))

## 5. So what — the bidding link

Attribution is only interesting because it changes what an impression is worth. Take your
Week 1 CVR model, multiply by each attribution scheme's credit to get an expected value
per impression, and show how the bid distribution shifts.

The Criteo paper's whole point is that attribution changes bidding *efficiency*. You now
have the pieces to show it, and Week 8 will plug these values into an actual auction.

In [ ]:
# TODO: value = p_conversion * credit_share; compare bid distributions across schemes

---
## Log the results

Every model you tried, including the baseline and including the failures. `notes` is the
one sentence you would say out loud about the run — future-you assembles the write-up
from these, so write it now while you still remember why the run mattered.

In [ ]:
# registry.log_result(
#     week=6,
#     model="lightgbm_hashed_2^18",
#     metrics=metrics.evaluate(y_test, p_test),
#     dataset="attribution",
#     params=dict(n_bits=18, num_leaves=63, lr=0.05),
#     notes="beats LR by 0.011 AUC; most of the gain is from cat3 x cat7 interactions",
# )

print(registry.to_markdown(week=6))

---
## Write it up

Open `README.md` in this folder and fill in the three sections. Keep it to a page.

- **What I built** — one paragraph, no code.
- **What the numbers say** — paste the table above; say which comparison is the honest one.
- **What surprised me** — the part worth reading. If nothing surprised you, you probably
  did not stress the model hard enough.

Then commit:

```bash
git add week06_* results/
git commit -m "week 06: <the finding, not the task>"
```